# Анализ лояльности пользователей Яндекс Афиши

<b>Цель исследования</b>

Выявить ключевые факторы, влияющие на лояльность пользователей сервиса «Яндекс Афиша», и разработать практические рекомендации для отделов маркетинга и продукта по удержанию клиентов и увеличению частоты повторных покупок.

<b>Содержание проекта</b>

* [1. Загрузка данных и их предобработка](#chapter1)
    * [SQL-запрос, выгружающий в датафрейм pandas данные](#chapter11)
    * [Изучение общей информации о выгруженных данных](#chapter12)
* [2. Предобработка данных](#chapter2)
    * [Конвертация валют](#chapter21)
    * [Очистка данных и обработка аномалий](#chapter22)
* [3. Создание профиля пользователя](#chapter3)
    * [Профиль пользователя](#chapter31)
    * [Оценка репрезентативности и валидация профилей](#chapter32)
* [4. Исследовательский анализ данных](#chapter4)
    * [4.1. Исследование признаков первого заказа и их связи с возвращением на платформу](#chapter41)
        * [Распределение пользователей по признакам](#chapter411)
        * [Анализ возврата пользователей](#chapter412)
        * [Проверка продуктовых гипотез](#chapter413)
    * [4.2. Исследование поведения пользователей через показатели выручки и состава заказа](#chapter42)
        * [Связь между средней выручкой сервиса с заказа и повторными заказами](#chapter421)
        * [Сравнение распределений по средней выручке с заказа в двух группах пользователей](#chapter422)
        * [Анализ влияния среднего количества билетов в заказе на вероятность повторной покупки](#chapter423)
    * [4.3. Исследование временных характеристик первого заказа и их влияния на повторные покупки](#chapter43)
        * [Влияние дня недели первой покупки на поведение пользователей](#chapter431)
        * [Влияние среднего интервала между заказами влияет на удержание клиентов](#chapter432)
    * [4.4. Корреляционный анализ количества покупок и признаков пользователя](#chapter44)
* [5. Общий вывод и рекомендации](#chapter5)
* [6. Финализация проекта и публикация в Git](#chapter6)

## Этапы выполнения проекта

### 1. Загрузка данных и их предобработка

---

**Задача 1.1:** Напишите SQL-запрос, выгружающий в датафрейм pandas необходимые данные. Используйте следующие параметры для подключения к базе данных `data-analyst-afisha`:

Для выгрузки используйте запрос из предыдущего урока и библиотеку SQLAlchemy.

Выгрузка из базы данных SQL должна позволить собрать следующие данные:

- `user_id` — уникальный идентификатор пользователя, совершившего заказ;
- `device_type_canonical` — тип устройства, с которого был оформлен заказ (`mobile` — мобильные устройства, `desktop` — стационарные);
- `order_id` — уникальный идентификатор заказа;
- `order_dt` — дата создания заказа (используйте данные `created_dt_msk`);
- `order_ts` — дата и время создания заказа (используйте данные `created_ts_msk`);
- `currency_code` — валюта оплаты;
- `revenue` — выручка от заказа;
- `tickets_count` — количество купленных билетов;
- `days_since_prev` — количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено;
- `event_id` — уникальный идентификатор мероприятия;
- `service_name` — название билетного оператора;
- `event_type_main` — основной тип мероприятия (театральная постановка, концерт и так далее);
- `region_name` — название региона, в котором прошло мероприятие;
- `city_name` — название города, в котором прошло мероприятие.

---


In [ ]:
import os
from dotenv import load_dotenv

import pandas as pd
import numpy as np
from sqlalchemy import create_engine 

# для визуализации
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors

# Загружаем библиотеку для расчёта коэффициента корреляции phi_k
from phik import phik_matrix

from difflib import SequenceMatcher

In [ ]:
# Загружаем данные из скрытого файла .env
load_dotenv('.env')

# Достаем данные из переменных окружения
db_user = os.getenv('DB_USER')
db_password = os.getenv('DB_PASSWORD')
db_host = os.getenv('DB_HOST')
db_name = os.getenv('DB_NAME')

# Собираем строку подключения 
connection_string = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(connection_string)

<div class="alert alert-block alert-info">
<b>Функции и палитра</b>
</div>

In [ ]:
# единая палитра
colors = ['#f47e7a', '#b71f5c', '#621237', '#614758','#cf7471', '#895d5b', '#aa7498', '#ec7fc6']
font_color = '#525252'
accent_color = '#b71f5c' 
# градиент для графиков
my_cmap = mcolors.LinearSegmentedColormap.from_list("berry_corr", ["#ffffff", "#f47e7a", "#b71f5c"])

In [ ]:
def show_missing_stats(tmp0):
    """
    Функция для отображения статистики пропущенных значений в DataFrame.
    """
    missing_stats = pd.DataFrame({
        'Кол-во пропусков': tmp0.isnull().sum(),
        'Доля пропусков': tmp0.isnull().mean()
    })
    missing_stats = missing_stats[missing_stats['Кол-во пропусков'] > 0]
    
    if missing_stats.empty:
        return "Пропусков в данных нет"

    return (missing_stats.style.format({'Доля пропусков': '{:.4f}'}).background_gradient(cmap='coolwarm'))


In [ ]:
def normalize_categorical(df, columns):
    """
    Функция нормализации строк. Приводит строки к нижнему регистру, удаляет пробелы, 
    заменяет 'ё' на 'е' и текстовые пропуски на NaN.
    """
    df_clean = df.copy()
    missing_markers = ['none', 'nan', 'unknown', 'n/a', '', 'undefined', '-', ' ']
    
    for col in columns:
        if col in df_clean.columns:
            # Приведение к строке, нижний регистр, удаление пробелов
            df_clean[col] = df_clean[col].astype(str).str.lower().str.strip()
            
            # Замена 'ё' на 'е'
            df_clean[col] = df_clean[col].str.replace('ё', 'е', regex=False)
            
            # Замена текстовых пропусков на NaN
            df_clean[col] = df_clean[col].replace(missing_markers, np.nan)
            
    return df_clean

In [ ]:
def find_similar_cities_with_counts(df, column='city_name', threshold=0.85):
    """
    Ищет похожие названия и выводит их частоту в данных.
    Тот вариант, которого мало — вероятная опечатка.
    """
    # Считаем частоту каждого города
    counts = df[column].value_counts(dropna=False).to_dict()
    
    # Берем только уникальные названия, исключая NaN
    cities = [str(c) for c in counts.keys() if pd.notna(c)]
    similar_pairs = []
    
    for i in range(len(cities)):
        for j in range(i + 1, len(cities)):
            city1, city2 = cities[i], cities[j]
            
            ratio = SequenceMatcher(None, city1, city2).ratio()
            
            if ratio >= threshold:
                similar_pairs.append({
                    'city_1': city1, 'count_1': counts.get(city1, 0),
                    'city_2': city2, 'count_2': counts.get(city2, 0),
                    'ratio': round(ratio, 2)
                })
                
    if not similar_pairs:
        print("Похожих названий не найдено.")
    else:
        res_df = pd.DataFrame(similar_pairs)
        # Сортируем по схожести
        res_df = res_df.sort_values(by='ratio', ascending=False)
        
        print(f"Найдено {len(res_df)} подозрительных пар:")
        display(res_df)

In [ ]:
def get_ticket_segment(tickets):
    """
    Классифицирует пользователей по среднему количеству билетов в заказе.
    
    Сегментация основана на социальных паттернах поведения:
    - [1, 2): Одиночные посетители или редкие парные походы.
    - [2, 3): Пары (наиболее распространенный тип заказа).
    - [3, 5): Небольшие компании или семьи.
    - [5, +inf): Большие группы или организаторы мероприятий.
    
    Аргументы:
        tickets (float): Среднее количество билетов в профиле пользователя.
        
    Возвращает:
        str: Название сегмента с порядковым номером для корректной сортировки.
    """
    if 1 <= tickets < 2:
        return '1-2 билета'
    elif 2 <= tickets < 3:
        return '2-3 билета'
    elif 3 <= tickets < 5:
        return '3-5 билетов'
    else:
        return 'от 5 билетов'

In [ ]:
def get_order_segment(orders):
    """
    Классифицирует пользователей по количеству совершенных заказов.
    
    Сегментация помогает разделить аудиторию на уровни лояльности:
    - '1 заказ': Новые/разовые пользователи.
    - '2-4 заказа': Активные (лояльное ядро).
    - '5+ заказов': Суперактивные
    """
    if orders == 1:
        return '1 заказ'
    elif 2 <= orders <= 4:
        return '2-4 заказа'
    else:
        return '5+ заказов'

<div class="alert alert-block alert-info">
<b>Выгрузка данных</b>
</div>

In [ ]:
# запрос на выгрузку данных
query = '''
WITH set_config_precode AS (
  SELECT set_config('synchronize_seqscans', 'off', true)
)
SELECT 
    user_id, 
    device_type_canonical,
    order_id,
    created_dt_msk as order_dt,
    created_ts_msk as order_ts,
    currency_code,
    revenue,
    tickets_count,
    (created_dt_msk::date - LAG(created_dt_msk::date) OVER (PARTITION BY user_id ORDER BY created_dt_msk))::int AS days_since_prev,
    e.event_id,
    event_name_code as event_name,
    event_type_main,
    service_name,
    region_name,
    city_name
FROM 
    afisha.purchases p
LEFT JOIN afisha.events e ON p.event_id=e.event_id
LEFT JOIN afisha.city c ON e.city_id = c.city_id
LEFT JOIN afisha.regions r ON c.region_id=r.region_id
WHERE 
    device_type_canonical IN ('mobile', 'desktop')   
    AND event_type_main!='фильм'
ORDER BY 
    user_id
''' 

In [ ]:
# выгрузка данных
df = pd.read_sql_query(query, con=engine) 

In [ ]:
df.head()

---

**Задача 1.2:** Изучите общую информацию о выгруженных данных. Оцените корректность выгрузки и объём полученных данных.

Предположите, какие шаги необходимо сделать на стадии предобработки данных — например, скорректировать типы данных.

Зафиксируйте основную информацию о данных в кратком промежуточном выводе.

---

Основная информацию о датафрейме

In [ ]:
df.info()

<b> Промежуточный вывод:</b>


Датасет содержит 15 столбцов и 290_611 строк, в которых представлена информация о продажах билетов на разные мероприятия сервиса Яндекс.Афиша

После первичного анализа данных можно сделать следующие выводы:
- названия столбцов приемлемые, их изменять не нужно;
- есть пропуски в данных о количестве дней, прошедших со дня предыдущей покупки (столбец `days_since_prev`), доля пропусков 7.6%;
- это же поле должно иметь тип Int, но из-за пропусков у него тип float64; 
- судя по первому знакомству с данными, значения в столбцах соответствуют своему описанию.

---

###  2. Предобработка данных

Выполните все стандартные действия по предобработке данных:

---

**Задача 2.1:** Данные о выручке сервиса представлены в российских рублях и казахстанских тенге. Приведите выручку к единой валюте — российскому рублю.

Для этого используйте датасет с информацией о курсе казахстанского тенге по отношению к российскому рублю за 2024 год — `final_tickets_tenge_df.csv`. Его можно загрузить по пути `https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')`

Значения в рублях представлено для 100 тенге.

Результаты преобразования сохраните в новый столбец `revenue_rub`.

---


In [ ]:
# информация о курсе казахстанского тенге по отношению к российскому рублю за 2024 год
tenge = pd.read_csv('https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')

In [ ]:
tenge.info()

In [ ]:
tenge['data'] = pd.to_datetime(tenge['data'])

In [ ]:
tenge.head()

In [ ]:
# объединение данных по коду валюты и дате
df_merged = pd.merge(
    df, 
    tenge, 
    left_on=['order_dt', 'currency_code'],
    right_on=['data', 'cdx'],
    how='left'
)

df_merged = df_merged.drop(columns=['data', 'cdx'])

In [ ]:
# маска по строкам с валютой тенге
mask = df_merged['currency_code'] == 'kzt'
# конвертация валют
df_merged.loc[mask, 'revenue_rub'] = (df_merged['revenue'] / df_merged['nominal']) * df_merged['curs']

# Для рублей (или если курс не найден) - исходное значение
df_merged['revenue_rub'] = df_merged['revenue_rub'].fillna(df_merged['revenue'])

# удаляем лишние столбцы
df_merged = df_merged.drop(columns=['nominal', 'curs'])

In [ ]:
df_merged.head()

In [ ]:
# удаляем лишние дф
del df, tenge

In [ ]:
# Рассчитываем стоимость одного билета
df_merged['ticket_price'] = df_merged['revenue_rub'] / df_merged['tickets_count']

In [ ]:
df_merged.info()

---

**Задача 2.2:**

- Проверьте данные на пропущенные значения. Если выгрузка из SQL была успешной, то пропуски должны быть только в столбце `days_since_prev`.
- Преобразуйте типы данных в некоторых столбцах, если это необходимо. Обратите внимание на данные с датой и временем, а также на числовые данные, размерность которых можно сократить.
- Изучите значения в ключевых столбцах. Обработайте ошибки, если обнаружите их.
    - Проверьте, какие категории указаны в столбцах с номинальными данными. Есть ли среди категорий такие, что обозначают пропуски в данных или отсутствие информации? Проведите нормализацию данных, если это необходимо.
    - Проверьте распределение численных данных и наличие в них выбросов. Для этого используйте статистические показатели, гистограммы распределения значений или диаграммы размаха.
        
        Важные показатели в рамках поставленной задачи — это выручка с заказа (`revenue_rub`) и количество билетов в заказе (`tickets_count`), поэтому в первую очередь проверьте данные в этих столбцах.
        
        Если обнаружите выбросы в поле `revenue_rub`, то отфильтруйте значения по 99 перцентилю.

После предобработки проверьте, были ли отфильтрованы данные. Если были, то оцените, в каком объёме. Сформулируйте промежуточный вывод, зафиксировав основные действия и описания новых столбцов.

---

<b>Проверка на пропуски</b>

In [ ]:
show_missing_stats(df_merged)

Пропуски наблюдаются в столбце `days_since_prev` и составляют 7.6% - это нормально, т.к. для каждого существует первая покупка, но не обязательно вторая. Так же этот столбец нужно преобразовать в целочисленный тип. Пропуски заполнять не будет никак.

<b>Преобразование типов данных</b>

In [ ]:
# типы данных
df_merged.dtypes

In [ ]:
# оптимизация типов данных
df_merged['tickets_count'] = pd.to_numeric(df_merged['tickets_count'], downcast='integer')
df_merged['days_since_prev'] = pd.to_numeric(df_merged['days_since_prev'], downcast='integer')
df_merged['event_id'] = pd.to_numeric(df_merged['event_id'], downcast='integer')

df_merged['revenue'] = pd.to_numeric(df_merged['revenue'], downcast='float')

<b>Обработка ошибок</b>

In [ ]:
# первоначальный размер датасета
len(df_merged)

<div class="alert alert-block alert-info">
<b>Категориальные значения</b>
</div>

In [ ]:
# Список столбцов с категориями
cat_cols = [
    'device_type_canonical', 'currency_code', 'event_type_main', 
    'event_name', 'service_name', 'region_name', 'city_name',
]

In [ ]:
# нормализация категориальных значений - обработка строковых данных
df_merged = normalize_categorical(df_merged, cat_cols)

In [ ]:
cat_cols = [
    'device_type_canonical', 'currency_code', 'event_type_main', 
     'service_name', 'region_name', 'city_name'
]
for col in cat_cols:
    display(f"--- Категории в {col} ---")
    # уникальные значения
    uniq_values = df_merged[col].unique()
    # Сортировка, используем key=str, чтобы корректно обработать NaN 
    sorted_values = sorted(uniq_values, key=lambda x: str(x))
    # Вывод
    display(sorted_values)

In [ ]:
# проверка городов на схожесть для выявления ошибок в написании
find_similar_cities_with_counts(df_merged, threshold=0.9)

Т.к. не все города - реальные географические названия, сложно судить о правильности написания. Исправлять не будем - для основной цели задачи эти данные не критичны

In [ ]:
df_merged.query('region_name in ("крутоводская область", "крутоводский регион", "речицкая область", "речицкий регион")')\
            ['region_name'].value_counts()

В целом, область и регион тоже могут иметь одинаковое название.

<div class="alert alert-block alert-info">
<b>Численные значения:</b> Выручка
</div>

In [ ]:
data_rev = pd.to_numeric(df_merged['revenue_rub'], errors='coerce').dropna()

В поле revenue_rub есть отрицательные числа и нулевая выручка.

In [ ]:
(data_rev == 0).sum()

In [ ]:
(data_rev < 0).sum()

In [ ]:
print(f"Количество строк с выручкой < 0: {(data_rev < 0).sum()}")
print(f"Количество строк с выручкой = 0: {(data_rev == 0).sum()}")
print(f"99-й перцентиль: {data_rev.quantile(0.99)}")

Отрицательные значения - могут обозначать возвраты, нулевые значения - могут быть билетами по промо-акциям или на бесплатные мероприятия. Это значение важно для подсчета количества билетов, но их нужно исключить при расчете среднего чека платящего пользователя

In [ ]:
plt.figure(figsize=(12, 6))

sns.histplot(data_rev, bins=100, kde=True, color=accent_color, edgecolor='white', alpha=0.8)
plt.title(f'Распределение выручки', fontsize=15, pad=20, color=font_color, fontweight='bold')
plt.xlabel('Выручка, руб.', fontsize=12, color=font_color)
plt.ylabel('Количество заказов', fontsize=12, color=font_color)

# Настройка сетки и границ
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
#plt.gca().tick_params(colors=font_color)

plt.show()

In [ ]:
del data_rev

In [ ]:
# Оставляем только заказы с положительной суммой для оценки платящих
df_clean = df_merged[df_merged['revenue_rub'] > 0].copy()

metrics_compare = df_clean[['revenue_rub', 'ticket_price']].describe()
display(metrics_compare)

Средний чек платящего пользователя (revenue_rub) - 567 руб., средняя цена билета платящего пользователя (ticket_price) - 205 руб., т.е. в среднем один заказ содержит 2.7 билета - люди редко ходят на мероприятия в одиночку

In [ ]:
# 99-й перцентиль для значений выручки и стоимости билетов платящих пользователей
rev_limit = df_clean['revenue_rub'].quantile(0.99)
price_limit = df_clean['ticket_price'].quantile(0.99)

In [ ]:
# Визуализация
plt.rcParams.update({'text.color': font_color, 'axes.labelcolor': font_color})

sns.set_theme(style="whitegrid")
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Выручка по заказу
sns.boxplot(x=df_clean['revenue_rub'], ax=ax1, color=accent_color, fliersize=4, 
            linewidth=2, medianprops={'color': 'white', 'linewidth': 2})
ax1.set_title(f'Выручка для платящих пользователей за весь заказ\nMax={df_clean["revenue_rub"].max():.2f} руб.', 
              fontsize=16, fontweight='bold', pad=15)
ax1.set_xlabel('Сумма заказа, руб.', fontsize=13, fontweight='bold')

# Стоимость одного билета
sns.boxplot(x=df_clean['ticket_price'], ax=ax2, color=colors[7], fliersize=4, 
            linewidth=2, medianprops={'color': 'white', 'linewidth': 2})
ax2.set_title(f'Стоимость одного билета для платящих пользователей\nMax={df_clean["ticket_price"].max():.2f} руб.', 
              fontsize=16, fontweight='bold', pad=15)
ax2.set_xlabel('Цена билета, руб.', fontsize=13, fontweight='bold')


for ax in [ax1, ax2]:
    ax.grid(axis='x', linestyle='-', alpha=0.2, color=font_color)
    ax.tick_params(labelsize=11, colors=font_color)
    sns.despine(ax=ax, left=True)

plt.tight_layout()
plt.show()

In [ ]:
# Визуализация
sns.set_theme(style="whitegrid")
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Выручка по заказу
sns.boxplot(x=df_clean['revenue_rub'], ax=ax1, color=accent_color, fliersize=3, 
            linewidth=2, medianprops={'color': 'white', 'linewidth': 2})
ax1.set_title(f'Выручка для платящих пользователей за весь заказ (99-перцентиль)\nMax={df_clean["revenue_rub"].max():.2f} руб.', 
              fontsize=14, fontweight='bold', pad=15)
ax1.set_xlabel('Сумма заказа, руб.', fontsize=12)
ax1.set_xlim(-100, rev_limit * 1.5) 

# Стоимость одно билета
sns.boxplot(x=df_clean['ticket_price'], ax=ax2, color=colors[7], fliersize=3, 
            linewidth=2, medianprops={'color': 'white', 'linewidth': 2})
ax2.set_title(f'Стоимость одного билета для платящих пользователей (99-перцентиль)\nMax={df_clean["ticket_price"].max():.2f} руб.', 
              fontsize=14, fontweight='bold', pad=15)
ax2.set_xlabel('Цена билета, руб.', fontsize=12)
ax2.set_xlim(-50, price_limit * 1.5)

for ax in [ax1, ax2]:
    ax.grid(axis='x', linestyle='--', alpha=0.3, color=font_color)
    ax.tick_params(colors=font_color)
    sns.despine(ax=ax, left=True)

plt.tight_layout()
plt.show()

Ящики расположены ближе к левой стороне, это означает, что большинство заказов - недорогие. Медианная стоимость билета 156 р., а заказа - 364 р., т.е. половина всех заказов ниже этой суммы. Минимальная ненулевая стоимость билета - 2 копейки (выглядит подозрительно)

In [ ]:
# Фильтруем подозрительно дешевые билеты
penniless_tickets = df_clean[df_clean['ticket_price'] < 10].copy()
# Общее количество заказов с положительной выручкой

# Расчет доли
share = (len(penniless_tickets) / len(df_clean)) * 100

print(f"Общее количество ненулевых заказов: {len(df_clean)}")
print(f"Доля 'копеечных' заказов: {share:.2f}%")

print(f"Количество билетов дешевле 10 рублей: {len(penniless_tickets)}")
print(f"Минимальная цена: {penniless_tickets['ticket_price'].min()} руб.")

In [ ]:
penniless_tickets.head()

In [ ]:
# Смотрим на распределение по сервисам и категориям
print("\nКто продает билеты за копейки:")
display(penniless_tickets.groupby(['service_name', 'event_type_main']).agg({
    'order_id': 'count',
    'ticket_price': ['min', 'max', 'mean']
}))

Основная масса недорогих билетов падает в категорию "Другое"

Доля заказов до 10 р. составляет 3.48% данных, в данных присутствуют выбросы, поэтому датасет фильтруем по 99 перцентилю

In [ ]:
df_analysis = df_clean[df_clean['revenue_rub'] <= rev_limit].copy()

In [ ]:
# Проверим сколько удалено строк датасета
print(" Было строк в исходном датасете", len(df_merged),
      '\n', "Осталось строк в датасете после первого этапа фильтрации", len(df_analysis),
      '\n', "Удалено строк в датасете после обработки", len(df_merged)-len(df_analysis),
      '\n', "Процент потерь", round((len(df_merged)-len(df_analysis))/len(df_merged)*100, 2))#

<div class="alert alert-block alert-info">
<b>Численные значения:</b> Количество билетов в заказе
</div>

Т.к. главное для бизнеса - выручка, дальше будем оценивать также заказы с ненулевой стоимостью билетов. 

In [ ]:
data_tickets = pd.to_numeric(df_analysis['tickets_count'], errors='coerce').dropna()
print(f"Статистика для tickets_count:")
display(data_tickets.describe())

In [ ]:
plt.figure(figsize=(12, 6))

sns.countplot(x=data_tickets, color=accent_color, edgecolor='white', alpha=0.9)
plt.title('Распределение количества билетов в одном заказе', fontsize=15, pad=20, color=font_color, fontweight='bold')
plt.xlabel('Кол-во билетов', fontsize=12, color=font_color)
plt.ylabel('Количество заказов', fontsize=12, color=font_color)

# Настройка сетки и осей
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().tick_params(colors=font_color)

plt.show()

Самое частое значение (мода) — 3 билета, на втором месте — 2 билета. Одиночные походы (1 билет) встречаются реже, чем походы втроем. Значительная доля заказов на 4–5 билетов указывает на популярность семейных мероприятий или компаний друзей. После 6 билетов идет резкий спад — это естественный предел для «обычного» пользователя.

Заказ на 47 билетов (и редкие значения 19, 27, 37 на оси X) — это явно не обычный клиент. Это либо корпоративные закупки, либо перекупщики, либо школьные экскурсии.

Посмотрим, на что покупают по 10+ билетов:

In [ ]:
# Изучаем заказы, где больше 10 билетов
bulk_orders = df_analysis[df_analysis['tickets_count'] > 10]

print(f"Количество 'оптовых' заказов: {len(bulk_orders)}")
print("Топ мероприятий для оптовых покупок:")
display(bulk_orders.groupby('event_type_main')['tickets_count'].agg(['count', 'sum', 'max']))

Спорт (max = 47): рекордсмен по количеству билетов в одном чеке. Скорее всего, это фанатские объединения или закупка на корпоративный выезд.

Театр (max = 19): групповые походы в театр (например, школьные/студенческие группы) — обычная практика.

Другое (count = 27): здесь сосредоточено больше всего крупных заказов. Вероятно, это специфические мероприятия (экскурсии, мастер-классы), на которые приходят организованными группами.

In [ ]:
# боксплот с ограничением до 15 заказов
plt.figure(figsize=(12, 5))
sns.boxplot(x=df_analysis['tickets_count'], 
            color=accent_color, 
            linewidth=2, 
            fliersize=4,
            medianprops={'color': 'white', 'linewidth': 2})

plt.title('Анализ выбросов: Количество билетов в одном заказе', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Количество билетов, шт.', fontsize=13)

# Ограничим обзор до 15
plt.xlim(0.5, 15.5) 
plt.xticks(range(1, 16))

sns.despine(left=True)
plt.grid(axis='x', linestyle='--', alpha=0.3)

plt.show()

«Ящик» крайне компактный. Это говорит о высокой предсказуемости поведения: 50% всех заказов — это либо 2, либо 3 билета. «Усы» показывают, что почти весь основной поток клиентов укладывается в этот интервал. 

Выбросы (5–15 шт.): каждая точка здесь — это большая компания. Видно, что такие заказы идут регулярно (точки стоят плотно), но они составляют статистическое меньшинство.

С выбросами делать ничего не будем, в дальнейшем такие многичесленные покупки можно объединить в одну категорию

<div class="alert alert-block alert-info">
<b>Численные значения:</b> Интервал между покупками
</div>

Каждый пользователь совершил первую покупку, но не каждый - вторую. Тех, кто совершил только первую покупку выделим в отдельную группу

In [ ]:
total_orders = len(df_analysis)
one_time_users = df_analysis['days_since_prev'].isna().sum()
returning_orders = df_analysis['days_since_prev'].notna().sum()

# Доля вернувшихся пользователей
retention_rate = (returning_orders / total_orders) * 100

In [ ]:
# сохраняем отдельно вернувшихся
returning_users_data = pd.to_numeric(df_analysis['days_since_prev'], errors='coerce').dropna()

In [ ]:
print(f"Всего заказов: {total_orders:,}")
print(f"Заказы новых пользователей (первые): {one_time_users:,} ({one_time_users/total_orders:.1%})")
print(f"Повторные заказы (вернувшиеся): {returning_orders:,} ({retention_rate:.1f}%)")

print(f"\nСтатистика интервалов между покупками (в днях):")
display(returning_users_data.describe())

In [ ]:
# Данные
labels = ['Новые пользователи', 'Повторные заказы']
sizes = [one_time_users, returning_orders]
# Цвета из палитры
plot_colors = [colors[0], accent_color] 


plt.figure(figsize=(8, 8))

# Круговая диаграмма
patches, texts, autotexts = plt.pie(
    sizes, labels=labels, autopct='%1.1f%%', 
    startangle=140, colors=plot_colors, 
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 12, 'fontweight': 'bold'}
)

for autotext in autotexts:
    autotext.set_color('white')

plt.title('Соотношение новых и повторных заказов', fontsize=16, fontweight='bold', pad=20)
plt.show()

Так же пользователь мог совершить покупку в тот же день

In [ ]:
total_returning = len(returning_users_data)
same_day_count = (returning_users_data == 0).sum()
later_day_count = (returning_users_data > 0).sum()

same_day_share = (same_day_count / total_returning) * 100
later_day_share = (later_day_count / total_returning) * 100

In [ ]:
print(f"Покупок в тот же день (0 дней): {same_day_count:,} ({same_day_share:.1f}%)")
print(f"Покупок в другие дни (1+ день): {later_day_count:,} ({later_day_share:.1f}%)")

In [ ]:
plt.figure(figsize=(10, 6))

labels = ['В тот же день', 'В другие дни']
values = [same_day_count, later_day_count]

plt.bar(labels, values, color=[accent_color, colors[4]], width=0.6)
# Добавляем подписи процентов
for i, val in enumerate(values):
    share = (val / total_returning) * 100
    plt.text(i, val + 2000, f'{share:.1f}%', ha='center', va='bottom', 
             fontweight='bold', color=font_color)

plt.title('Типы повторных покупок', fontsize=15, fontweight='bold', pad=20, color=font_color)
plt.ylabel('Количество заказов', color=font_color)
plt.grid(axis='y', linestyle='--', alpha=0.3)
sns.despine(left=True)

plt.show()

Почти 7 из 10 повторных покупок совершаются в тот же день. Это говорит о том, что пользователи не «возвращаются в сервис спустя время», а скорее «доукомплектовывают» свой досуг прямо сейчас. Технический нюанс: возможно, система дробит крупные заказы на несколько транзакций.

Проверим заказы в тот же день

In [ ]:
df_sorted = df_analysis.sort_values(['user_id', 'order_ts'])
# Считаем разницу в секундах между текущим и предыдущим заказом пользователя
df_sorted['sec_diff'] = df_sorted.groupby('user_id')['order_ts'].diff().dt.total_seconds()

In [ ]:
# возможные технические дубликаты, быстрые покупки по времени
potential_technical_dupes = df_sorted[
    (df_sorted['sec_diff'] > 0) & 
    (df_sorted['sec_diff'] <= 10) & 
    (df_sorted['event_id'] == df_sorted.groupby('user_id')['event_id'].shift(1)) &
    (df_sorted['revenue_rub'] == df_sorted.groupby('user_id')['revenue_rub'].shift(1))
]
print(f"Найдено подозрительных дублей: {len(potential_technical_dupes)}")

# примеры пользователей
example_users = potential_technical_dupes['user_id'].unique()[:5]

display(df_sorted[df_sorted['user_id'].isin(example_users)][
    ['user_id', 'order_id', 'order_ts', 'event_name', 'revenue_rub', 'sec_diff']
].sort_values(['user_id', 'order_ts']))

In [ ]:
# пример заказов с разницей в несколько минут
df_sorted.query('user_id=="0eb26a9ddcee1dd" and event_name=="e252d33a-2394-44b8-b11d-f0709a6f0260"').sort_values(by='order_ts')

В целом, наверное, ок, но тут нужно уточнять у тех, кто данные собирает, о возможном разбиении транзакций.

Для того, чтобы оценить "честный"  Retention пользователей, проанализируем тех, кто возвращался в другие дни

In [ ]:
# Только возвраты спустя 1 день и более
clean_retention = returning_users_data[returning_users_data > 0].astype(float)

print(f"Количество осознанных возвратов: {len(clean_retention):,}")
print(f"Доля от всех повторных заказов: {len(clean_retention)/len(returning_users_data):.1%}")

display(clean_retention.describe())

In [ ]:
plt.figure(figsize=(14, 7))

# Ограничим 90 днями (квартал)
data_plot = clean_retention[clean_retention <= 90]

# Гистограмма
sns.histplot(data_plot, bins=90, kde=True, color=accent_color, edgecolor='white', alpha=0.8)

plt.title('Распределение осознанных возвратов (от 1 до 90 дней)', 
          fontsize=16, fontweight='bold', pad=20, color=font_color)
plt.xlabel('Количество дней между покупками', fontsize=13, color=font_color)
plt.ylabel('Количество заказов', fontsize=13, color=font_color)

# Настройка сетки
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.xticks(range(0, 91, 7)) 
sns.despine(left=True)

plt.show()

Половина осознанных возвратов происходит в течение первых трех дней после покупки (медиана 3 дня). 75% клиентов возвращаются в течение 10 дней. Если клиент не купил ничего нового за полторы недели, вероятность его возврата падает почти до нуля. На графике нет всплесков на 7, 14 или 21 днях. Это значит, что аудитория не привязана к ритму выходных (суббота-воскресенье). Покупки совершаются хаотично в любой день недели.

<b> Промежуточный вывод:</b>

1. Категориальные переменные
- Нормализация: Все текстовые поля приведены к нижнему регистру, удалены лишние пробелы, буква «ё» заменена на «е» для устранения дублей в названиях городов и мероприятий.
- Обработка пропусков: Скрытые маркеры отсутствия данных (none, unknown, ' ') заменены на  NaN.
2. Числовые переменные
- Оптимизированы типы данных
- Удалена 381 строка с отрицательной выручкой
- Доля заказов до 10 р. составляет 3.48% данных, также в данных присутствуют выбросы по выручке, поэтому датасет отфильтрован по 99 перцентилю по полю revenue_rub
3. Анализ числовых столбцов
- Выручка за заказ (revenue_rub): медианный чек составляет около 360–370 рублей. 99% всех заказов укладываются в сумму до 2600–2800 рублей. Все, что выше, — это редкие VIP-покупки или корпоративные заказы, которые вынесены за скобки для анализа типичного поведения.
- Стоимость одного билета (price_per_ticket): рассчитана как выручка / количество билетов. Реальная цена входа на мероприятие крайне низкая — медиана около 150–160 рублей. Это говорит о том, что основной объем продаж приходится на кинотеатры и недорогие выставки.
- Количество билетов в заказе (tickets_count): средний заказ содержит 2.7 билета, а самое частое значение (мода) — 3 билета. Одиночные походы в разы менее популярны, чем групповые. Большинство пользователей (99.9%) покупают не более 10–12 билетов за раз (максимальное значение - 47)
- Интервал между покупками (days_since_prev): выявлено два типа лояльности: 1) происходит в тот же день (0 дней) 68% - люди докупают билеты порциями. 2) Осознанный возврат 32% - имеет очень короткий цикл — медиана всего 3 дня.

После всех фильтраций размер датасета уменьшился на 3%

---

### 3. Создание профиля пользователя

В будущем отдел маркетинга планирует создать модель для прогнозирования возврата пользователей. Поэтому сейчас они просят вас построить агрегированные признаки, описывающие поведение и профиль каждого пользователя.

---

**Задача 3.1.** Постройте профиль пользователя — для каждого пользователя найдите:

- дату первого и последнего заказа;
- устройство, с которого был сделан первый заказ;
- регион, в котором был сделан первый заказ;
- билетного партнёра, к которому обращались при первом заказе;
- жанр первого посещённого мероприятия (используйте поле `event_type_main`);
- общее количество заказов;
- средняя выручка с одного заказа в рублях;
- среднее количество билетов в заказе;
- среднее время между заказами.

После этого добавьте два бинарных признака:

- `is_two` — совершил ли пользователь 2 и более заказа;
- `is_five` — совершил ли пользователь 5 и более заказов.

**Рекомендация:** перед тем как строить профиль, отсортируйте данные по времени совершения заказа.

---


В предыдущем разеделе выручка исслевадовалась для платящих пользователей, однако для анализа возврата пользователей нужно учитывать первые (возможно, промо) покупки за 0 рублей.
Применим фильтры, обоснованные выше, к основному датафрейму:
- убираем заказы с отрицательной выручкой
- на уровне транзакций оставляем выручку за заказ и количество билетов на уровне 99 перцентиля
- первые промо-заказы (менее 10 рублей) - не удаляем

In [ ]:
# границы 99 перцентиля для фильтрации
rev_threshold = df_merged['revenue_rub'].quantile(0.99)
tickets_threshold = df_merged['tickets_count'].quantile(0.99)

In [ ]:
# Применяем фильтрацию к основному датасету
df_cleaned = df_merged[
    (df_merged['revenue_rub'] >= 0) & # Оставляем 0 (промо), убираем отрицательные
    (df_merged['revenue_rub'] <= rev_threshold) & 
    (df_merged['tickets_count'] <= tickets_threshold) &
    (df_merged['tickets_count'] > 0)
].copy()

In [ ]:
# Расчет доли удаленных данных
lost_data = 1 - (len(df_cleaned) / len(df_merged))

In [ ]:
print(f"Размер датасета до фильтрации: {len(df_merged):,} строк")
print(f"Размер датасета после фильтрации: {len(df_cleaned):,} строк")
print(f"Удалено данных: {len(df_merged) - len(df_cleaned):,} строк")
print(f"Доля удаленных данных: {lost_data:.2%}")

In [ ]:
# сортировка
df_analysis = df_cleaned.sort_values(['user_id', 'order_ts'])

In [ ]:
# Агрегация основных признаков
user_profiles = df_analysis.groupby('user_id').agg(
    first_order_date=('order_dt', 'first'),
    last_order_date=('order_dt', 'last'),
    first_device=('device_type_canonical', 'first'),
    first_region=('region_name', 'first'),
    first_service=('service_name', 'first'),
    first_genre=('event_type_main', 'first'),
    total_orders=('order_id', 'count'),
    avg_revenue=('revenue_rub', 'mean'),      # средняя выручка с одного заказа в рублях
    avg_tickets=('tickets_count', 'mean'),    # среднее количество билетов в заказе
    avg_time_diff=('days_since_prev', 'mean'), # среднее только для вернувшихся
    # Добавляем признаки для работы с промо-заказами
    first_order_revenue=('revenue_rub', 'first'), # ВЫРУЧКА ПЕРВОГО ЗАКАЗА
    #promo_orders_cnt=('revenue_rub', lambda x: (x < 10).sum()),
    #real_orders_cnt=('revenue_rub', lambda x: (x >= 10).sum())
).reset_index()

In [ ]:
# разметка первого заказа по типу промо/платный
user_profiles['is_promo_first'] = (user_profiles['first_order_revenue'] < 10).map(
    {True: 'Промо (<10р)', False: 'Платный (>=10р)'}
)

In [ ]:
user_profiles.head()

In [ ]:
#user_profiles[['promo_orders_cnt', 'real_orders_cnt']] = user_profiles[['promo_orders_cnt', 'real_orders_cnt']].astype(int)

In [ ]:
# Доля промо-заказов в профиле
#user_profiles['promo_share'] = user_profiles['promo_orders_cnt'] / user_profiles['total_orders']

In [ ]:
# Добавление бинарных признаков
user_profiles['is_two'] = (user_profiles['total_orders'] >= 2).astype(int)
user_profiles['is_five'] = (user_profiles['total_orders'] >= 5).astype(int)

In [ ]:
print(f"Количество уникальных профилей: {len(user_profiles)}")
display(user_profiles.head())

---

**Задача 3.2.** Прежде чем проводить исследовательский анализ данных и делать выводы, важно понять, с какими данными вы работаете: насколько они репрезентативны и нет ли в них аномалий.

Используя данные о профилях пользователей, рассчитайте:

- общее число пользователей в выборке;
- среднюю выручку с одного заказа;
- долю пользователей, совершивших 2 и более заказа;
- долю пользователей, совершивших 5 и более заказов.

Также изучите статистические показатели:

- по общему числу заказов;
- по среднему числу билетов в заказе;
- по среднему количеству дней между покупками.

По результатам оцените данные: достаточно ли их по объёму, есть ли аномальные значения в данных о количестве заказов и среднем количестве билетов?

Если вы найдёте аномальные значения, опишите их и примите обоснованное решение о том, как с ними поступить:

- Оставить и учитывать их при анализе?
- Отфильтровать данные по какому-то значению, например, по 95-му или 99-му перцентилю?

Если вы проведёте фильтрацию, то вычислите объём отфильтрованных данных и выведите статистические показатели по обновлённому датасету.

In [ ]:
# Основные показатели
total_users = len(user_profiles)
avg_revenue_total = user_profiles['avg_revenue'].mean()
share_is_two = user_profiles['is_two'].mean() * 100
share_is_five = user_profiles['is_five'].mean() * 100

In [ ]:
print(f"Общее число пользователей: {total_users:,}")
print(f"Средняя выручка с одного заказа: {avg_revenue_total:.2f} руб.")
print(f"Доля пользователей с 2+ заказами: {share_is_two:.2f}%")
print(f"Доля пользователей с 5+ заказами: {share_is_five:.2f}%")

In [ ]:
display(user_profiles[['total_orders', 'avg_tickets', 'avg_time_diff']].describe())

Анализ:
1. Поле total_orders: максимум 10155 заказов, минимум 1, среднее - 13, в то время как медиана - 2, стандартное отклонение 121. Все это говорит о том, что либо это технический аккаунт либо это бот-перекупщик, т.к. обычные люди столько заказов не делают. Такого пользователя, конечно, нужно удалить из данных
2. Поле avg_time_diff: медиана - 8 дней, а среднее - 16 - обычный клиент возвращается примерно раз в неделю или две.
3. Поле avg_tickets=2.74 выглядит вполне адекватным


In [ ]:
# порог 99-го перцентиля
orders_p99 = user_profiles['total_orders'].quantile(0.99)
print(f"Порог 99% по заказам: {orders_p99}")

In [ ]:
# Фильтруем профили
user_profiles_final = user_profiles[user_profiles['total_orders'] <= orders_p99].copy()

print(f"Удалено аномальных профилей: {len(user_profiles) - len(user_profiles_final)}")
print(f"Доля: {round(1- len(user_profiles_final) / len(user_profiles), 2 ) * 100}%")
display(user_profiles_final[['total_orders', 'avg_tickets', 'avg_time_diff']].describe())

<b>Промежуточные выводы</b>

Данные репрезентативны и готовы к исследовательскому анализу. После удаления 1% экстремальных выбросов выборка стала однородной. Средний портрет пользователя: 2-3 заказа по 2.7 билета с возвратом каждые 8-16 дней.

---

### 4. Исследовательский анализ данных

Следующий этап — исследование признаков, влияющих на возврат пользователей, то есть на совершение повторного заказа. Для этого используйте профили пользователей.



#### 4.1. Исследование признаков первого заказа и их связи с возвращением на платформу

Исследуйте признаки, описывающие первый заказ пользователя, и выясните, влияют ли они на вероятность возвращения пользователя.

---

**Задача 4.1.1.** Изучите распределение пользователей по признакам.

- Сгруппируйте пользователей:
    - по типу их первого мероприятия;
    - по типу устройства, с которого совершена первая покупка;
    - по региону проведения мероприятия из первого заказа;
    - по билетному оператору, продавшему билеты на первый заказ.
- Подсчитайте общее количество пользователей в каждом сегменте и их долю в разрезе каждого признака. Сегмент — это группа пользователей, объединённых определённым признаком, то есть объединённые принадлежностью к категории. Например, все клиенты, сделавшие первый заказ с мобильного телефона, — это сегмент.
- Ответьте на вопрос: равномерно ли распределены пользователи по сегментам или есть выраженные «точки входа» — сегменты с наибольшим числом пользователей?

---


Расчет распределения по сегментам

In [ ]:
user_profiles_final.head()

In [ ]:
# Список признаков для анализа
features = ['first_genre', 'first_device', 'first_region', 'first_service', 'is_promo_first']
total_users = len(user_profiles_final)

In [ ]:
for col in features:
    print(f"\n--- Распределение по признаку: {col} ---")
    
    # Группировка и расчет
    stats = user_profiles_final[col].value_counts().reset_index()
    stats.columns = [col, 'users_count']
    stats['share'] = (stats['users_count'] / total_users * 100).round(2)
    
    display(stats.head(10)) # Выводим ТОП-10 для каждого признака

In [ ]:
# Промо

plt.figure(figsize=(10, 7))

# Параметры сегментации
# stats['is_promo_first'] - категории
# stats['users_count'] - значения
explode = (0, 0.2)  

patches, texts, autotexts = plt.pie(
    stats['users_count'], 
    labels=stats['is_promo_first'], 
    autopct='%1.2f%%', 
    startangle=160, 
    colors=[colors[4], accent_color], 
    explode=explode,
    wedgeprops={'edgecolor': 'white', 'linewidth': 3},
    pctdistance=0.75
)

# Стилизация подписей
for text in texts:
    text.set_color(font_color)
    text.set_weight('bold')
    text.set_fontsize(12)

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_weight('bold')
    autotext.set_fontsize(11)

plt.title('Структура первого входа: Платный vs Промо', 
          fontsize=16, fontweight='bold', color=font_color, pad=25)

plt.tight_layout()
plt.show()

In [ ]:
# Расчет данных
genre_counts = user_profiles_final['first_genre'].value_counts().head(7)
total_users = len(user_profiles_final)

plt.figure(figsize=(15, 7))

# График 1. Типы мероприятий
plt.subplot(1, 2, 1)
top_genres = user_profiles_final['first_genre'].value_counts()

ax = sns.barplot(x=top_genres.values, y=top_genres.index, palette=colors)
# Добавляем доли и количество в подписи
for i, p in enumerate(ax.patches):
    width = p.get_width()
    percentage = (width / total_users) * 100
    ax.text(width + 100, p.get_y() + p.get_height()/2, 
            f'{int(width):,} ({percentage:.1f}%)', 
            va='center', fontsize=11, color=font_color, fontweight='bold')

plt.title('Типы мероприятий "первого касания"', fontsize=14, fontweight='bold', color=font_color)
plt.xlabel('Количество пользователей', color=font_color)
plt.ylabel('') 

plt.grid(False)
sns.despine(left=True, bottom=True) 

# График 2. Устройства входа
plt.subplot(1, 2, 2)
device_stats = user_profiles_final['first_device'].value_counts()

pie_colors = [accent_color, colors[4]] 

patches, texts, autotexts = plt.pie(
    device_stats, 
    labels=device_stats.index, 
    autopct='%1.1f%%', 
    startangle=140, 
    colors=pie_colors,
    wedgeprops={'edgecolor': 'white', 'linewidth': 3}, 
    pctdistance=0.75
)

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_weight('bold')
    autotext.set_fontsize(11)

plt.title('Первое устройство', fontsize=14, fontweight='bold', color=font_color)

plt.tight_layout()
plt.show()

"Концерты" — главный магнит: 9.5 тыс.  человек (~44%) пришли на платформу именно ради концерта, это основная точка входа. "Другое" и "Театр" вместе забирают еще около 45%. Остальные категории (стендап, спорт и т.д.) суммарно дают менее 15% «точек входа».

Mobile доминирует (82.8%):сервис — это на 4/5 мобильная история. Любые маркетинговые кампании должны быть заточены под вертикальные экраны и быстрые покупки.

Распределение явно неравномерное: бизнес критически зависит от успеха концертных и театральных афиш, а также от типа устройств.

In [ ]:
plt.figure(figsize=(15, 12))

# График 3. Регионы первого заказа (Топ-5) 
plt.subplot(2, 1, 1)
region_counts = user_profiles_final['first_region'].value_counts().head(5)

ax1 = sns.barplot(x=region_counts.values, y=region_counts.index, palette=colors)

# Добавляем доли
for p in ax1.patches:
    width = p.get_width()
    percentage = (width / len(user_profiles_final)) * 100
    ax1.text(width + 50, p.get_y() + p.get_height()/2, 
            f'{int(width):,} ({percentage:.1f}%)', 
            va='center', fontsize=11, color=font_color, fontweight='bold')

plt.title('Топ-5 регионов первого заказа', fontsize=16, fontweight='bold', pad=20, color=font_color)
plt.xlabel('Количество пользователей', color=font_color)
plt.ylabel('')
plt.grid(False)
sns.despine(left=True, bottom=True)

# График 4. Билетные партнёры (Топ-7) 
plt.subplot(2, 1, 2)
service_counts = user_profiles_final['first_service'].value_counts().head(5)

ax2 = sns.barplot(x=service_counts.values, y=service_counts.index, palette=colors[::-1]) 

# Добавляем доли
for p in ax2.patches:
    width = p.get_width()
    percentage = (width / len(user_profiles_final)) * 100
    ax2.text(width + 50, p.get_y() + p.get_height()/2, 
            f'{int(width):,} ({percentage:.1f}%)', 
            va='center', fontsize=11, color=font_color, fontweight='bold')

plt.title('Топ-5 билетных партнёров первого заказа', fontsize=16, fontweight='bold', pad=20, color=font_color)
plt.xlabel('Количество пользователей', color=font_color)
plt.ylabel('')
plt.grid(False)
sns.despine(left=True, bottom=True)

plt.tight_layout(pad=3.0)
plt.show()

Региональное лидерство (Топ-5):
- Распределение крайне неравномерное. Один  Каменевский регион генерирует почти 33% (6 973 чел.) всех новых пользователей.
- Вместе с Североярской областью они обеспечивают половину (50.1%) всего притока, являясь главными рынками.
- Остальные регионы сильно отстают, что указывает на высокую географическую концентрацию интереса.


Билетные партнёры (Топ-5): рынок более фрагментирован, но явный лидер — «Билеты без проблем» (~24%). Почти каждая первая покупка совершается через них.

<b>Промежуточные выводы</b>

Пользователи распределены по сегментам крайне неравномерно. Есть чёткие точки входа — сегменты-лидеры, которые обеспечивают основной поток:
- Тип мероприятия: Концерты (абсолютный лидер - 44%).
- Устройство: Mobile (более 82% трафика).
- Регион: Каменевский регион (~33% пользователей).
- Партнёр: «Билеты без проблем» (~24% первых транзакций).
- Доля промо-входов (выручка первого заказа < 10 руб.) крайне мала — всего 2.54% (548 человек).

---

**Задача 4.1.2.** Проанализируйте возвраты пользователей:

- Для каждого сегмента вычислите долю пользователей, совершивших два и более заказа.
- Визуализируйте результат подходящим графиком. Если сегментов слишком много, то поместите на график только 10 сегментов с наибольшим количеством пользователей. Такое возможно с сегментами по региону и по билетному оператору.
- Ответьте на вопросы:
    - Какие сегменты пользователей чаще возвращаются на Яндекс Афишу?
    - Наблюдаются ли успешные «точки входа» — такие сегменты, в которых пользователи чаще совершают повторный заказ, чем в среднем по выборке?

При интерпретации результатов учитывайте размер сегментов: если в сегменте мало пользователей (например, десятки), то доли могут быть нестабильными и недостоверными, то есть показывать широкую вариацию значений.

---


In [ ]:
user_profiles_final.head()

In [ ]:
# Список признаков для анализа возврата
features = ['first_genre', 'first_device', 'first_region', 'first_service']
avg_retention = user_profiles_final['is_two'].mean() # Среднее по всей базе

In [ ]:
plt.figure(figsize=(16, 18))
plt.suptitle('Анализ факторов лояльности и удержания пользователей', 
             fontsize=20, fontweight='bold', color=font_color, y=0.98)
for i, col in enumerate(features, 1):
    plt.subplot(2, 2, i)
    
    # Считаем Retention для каждого сегмента топ-10
    top_segments = user_profiles_final[col].value_counts().head(10).index
    df_top = user_profiles_final[user_profiles_final[col].isin(top_segments)]
    
    segment_retention = df_top.groupby(col)['is_two'].mean().sort_values(ascending=False)
    
    # Визуализация 
    sns.barplot(x=segment_retention.values, y=segment_retention.index, palette=colors)
    
    # Линия среднего значения для сравнения
    plt.axvline(avg_retention, color=accent_color, linestyle='--', label=f'Среднее ({avg_retention:.1%})')
    
    # Подписи процентов на барах
    for j, val in enumerate(segment_retention.values):
        plt.text(val + 0.01, j, f'{val:.1%}', va='center', fontweight='bold', color=font_color)
    
    plt.title(f'Доля возврата (is_two) по {col}', fontsize=14, fontweight='bold', color=font_color)
    plt.xlabel('Retention Rate')
    plt.xlim(0, segment_retention.max() + 0.1)
    sns.despine()
plt.tight_layout(pad=4.0)
plt.show()

In [ ]:
# Отдельный анализ влияния промо-входа
plt.figure(figsize=(10, 5))
promo_retention = user_profiles_final.groupby('is_promo_first')['is_two'].mean().sort_values(ascending=False)

ax_promo = sns.barplot(x=promo_retention.values, y=promo_retention.index, palette=[colors[0], accent_color])

# Линия среднего для контекста
plt.axvline(avg_retention, color=accent_color, linestyle='--', alpha=0.6)

for i, val in enumerate(promo_retention.values):
    plt.text(val + 0.01, i, f'{val:.1%}', va='center', fontweight='bold', color=font_color, fontsize=12)

plt.title('Влияние типа входа на лояльность (Retention)', fontsize=15, fontweight='bold', color=font_color, pad=20)
plt.xlabel('Доля вернувшихся (is_two)')
plt.xlim(0, 1.0)
sns.despine(left=True)
plt.show()

1. Анализ по типам мероприятий (first_genre)
- Успешные точки входа (выше среднего): Выставки (64.4%) и Театр (63.4%). Это «интеллектуальные» точки входа. Люди, приходящие на выставки или спектакли, чаще всего становятся постоянными клиентами. Возможно, это более возрастная или вдумчивая аудитория, которая планирует свой досуг системно.
- Массовые, но средние: Концерты (61.8%). Несмотря на то, что это самый массовый сегмент входа, его Retention находится на уровне среднего. 
- Слабые точки входа: Спорт (55.8%) и Елки (55.8%) -  событийный досуг. Люди приходят на конкретный матч или на праздник раз в год и после этого «засыпают». У этих категорий самый низкий потенциал превращения новичка в постоянного покупателя.
2. Анализ по устройствам (first_device)
- Desktop (63.8%) удерживает людей лучше, чем Mobile. Покупка с компьютера обычно более осознанная, часто связана с выбором мест в театре или на концерте. Это более «лояльный» интерфейс.
- Mobile (60.8%) — это импульсивные покупки «на бегу». Доля возврата здесь ниже среднего. Учитывая, что 83% людей заходят с мобильных, но Desktop удерживает их лучше, маркетингу стоит подумать, как упростить повторную покупку в приложении/мобильной версии.
3. Анализ по регионам (first_region)
- Абсолютный лидер - Шанырский регион. 67% людей, купивших там первый билет, возвращаются снова. Это значительно выше среднего уровня (пунктирная линия).
- Стабильные лидеры: Светополянский округ и Широковская область также показывают отличный Retention (64.5%+).
- Зона риска: Озернинский край (55.2%) и Малиновоярский округ (55.8%). Несмотря на то, что это крупные регионы, пользователи оттуда «отваливаются» гораздо чаще (гипотеза: возможно, в этих регионах мало регулярных событий, и люди приходят только на редкие гастроли.)
4. Анализ по партнёрам (first_service)
- Успешные точки входа: "Край билетов" (65.2%) и "Дом культуры" (64.6%). Клиенты этих операторов самые лояльные. Это может быть связано с качеством сервиса или спецификой репертуара (например, более «театральные» или «семейные» площадки).
- Массовый сегмент: лидер по привлечению «Билеты без проблем» имеет самый низкий Retention среди ТОП-10 (60.3%). Этот партнёр приводит больше всего «новичков» (24%), но они же хуже всего удерживаются. 
5. Анализ по типу входа платный / промо
- Промо-вход (<10р) демонстрирует  высокий уровень лояльности — 64.2%.



<b>Промежуточный вывод</b>
1. Самые лояльные сегменты: Это пользователи из Шанырского региона, покупающие первый билет через «Край билетов» на Выставки или Театр через Desktop. Это идеальный профиль «возвратного» клиента.
2. Успешные «точки входа»: Шанырский регион и Светополянский округ — это территории, где маркетинговые инвестиции окупаются лучше всего за счёт повторных покупок.
3. Парадокс массовости: Самые крупные сегменты по привлечению (Концерты, Mobile, «Билеты без проблем») показывают Retention ниже среднего. Компании нужно работать над тем, как удерживать эту огромную массу людей после первого «хайпового» события.
4. Промо-вход - работает на удержание

---

**Задача 4.1.3.** Опираясь на выводы из задач выше, проверьте продуктовые гипотезы:

- **Гипотеза 1.** Тип мероприятия влияет на вероятность возврата на Яндекс Афишу: пользователи, которые совершили первый заказ на спортивные мероприятия, совершают повторный заказ чаще, чем пользователи, оформившие свой первый заказ на концерты.
- **Гипотеза 2.** В регионах, где больше всего пользователей посещают мероприятия, выше доля повторных заказов, чем в менее активных регионах.

---

<b> Проверка гипотезы 1. Тип мероприятия влияет на вероятность возврата на Яндекс Афишу: пользователи, которые совершили первый заказ на спортивные мероприятия, совершают повторный заказ чаще, чем пользователи, оформившие свой первый заказ на концерты.</b>

- Retention Rate (is_two) для Спорта = 55.8%
- Retention Rate (is_two) для Концертов = 61.8%

Гипотеза опровергнута: пользователи, купившие первый билет на концерт, возвращаются на 7 процентных пунктов чаще. «Спорт» оказался самым слабым жанром с точки зрения удержания среди всех представленных категорий.

<b>Проверка гипотезы 2. В регионах, где больше всего пользователей посещают мероприятия, выше доля повторных заказов, чем в менее активных регионах. </b>

- Каменевский регион (самый массовый, 32.8% всех входов) имеет Retention = 62.4%.
- Шанырский регион (не самый массовый по входу - не входит в топ-5, но лидер по качеству) имеет Retention = 67.3%.
- Озернинский край (крупная точка входа, 3.1%) имеет самый низкий Retention = 55.3%.

Гипотеза опровергнута (частично). Прямой связи «чем больше людей, тем выше лояльность» не наблюдается. Абсолютный лидер по привлечению (Каменевский регион) находится лишь в середине списка по возврату. В то же время Шанырский регион показывает феноменальный результат по удержанию, не являясь самым массовым. Лояльность в регионах больше зависит от местной инфраструктуры и репертуара, чем от общего количества пользователей.

<b>Промежуточный вывод</b>

Обе гипотезы не подтвердились:  массовость ≠ лояльность. Спорт приводит людей, но не удерживает их. Огромные регионы дают поток, но «качественные» лояльные ядра формируются в более специфических географических зонах.

---

#### 4.2. Исследование поведения пользователей через показатели выручки и состава заказа

Изучите количественные характеристики заказов пользователей, чтобы узнать среднюю выручку сервиса с заказа и количество билетов, которое пользователи обычно покупают.

Эти метрики важны не только для оценки выручки, но и для оценки вовлечённости пользователей. Возможно, пользователи с более крупными и дорогими заказами более заинтересованы в сервисе и поэтому чаще возвращаются.

---

**Задача 4.2.1.** Проследите связь между средней выручкой сервиса с заказа и повторными заказами.

- Постройте сравнительные гистограммы распределения средней выручки с билета (`avg_revenue_rub`):
    - для пользователей, совершивших один заказ;
    - для вернувшихся пользователей, совершивших 2 и более заказа.
- Ответьте на вопросы:
    - В каких диапазонах средней выручки концентрируются пользователи из каждой группы?
    - Есть ли различия между группами?

Текст на сером фоне:
    
**Рекомендация:**

1. Используйте одинаковые интервалы (`bins`) и прозрачность (`alpha`), чтобы визуально сопоставить распределения.
2. Задайте параметру `density` значение `True`, чтобы сравнивать форму распределений, даже если число пользователей в группах отличается.

---


Сравним среднюю выручку двух групп пользователей: is_two=0 - только один заказ, is_two=1 - вернувшиеся пользователи с более чем двумя заказами

In [ ]:
# Данные
one_order = user_profiles_final[user_profiles_final['is_two'] == 0]['avg_revenue']
multi_orders = user_profiles_final[user_profiles_final['is_two'] == 1]['avg_revenue']

In [ ]:
one_order.describe()

In [ ]:
multi_orders.describe()

In [ ]:
plt.figure(figsize=(15, 8))
# Построение гистограмм плотности с кривыми KDE
sns.histplot(one_order, bins=50, stat="density", color='#f47e7a', alpha=0.4, 
             label='Новые пользователи (1 заказ)', kde=True, edgecolor='none')
sns.histplot(multi_orders, bins=50, stat="density", color='#b71f5c', alpha=0.5, 
             label='Вернувшиеся пользователи (2+ заказа)', kde=True, edgecolor='none')

# Расчет средних значений
mean_one = one_order.mean()
mean_multi = multi_orders.mean()

# Вертикальные линии для средних
plt.axvline(mean_one, color='#f47e7a', linestyle='--', linewidth=2)
plt.axvline(mean_multi, color='#b71f5c', linestyle='--', linewidth=2)

# Аннотации для средних (исправлено с median на mean)
plt.text(mean_one + 30, 0.0022, f'Среднее (1): {mean_one:.0f} р.', color='#f47e7a', fontweight='bold')
plt.text(mean_multi + 30, 0.0020, f'Среднее (2+): {mean_multi:.0f} р.', color='#b71f5c', fontweight='bold')

# Оформление
plt.title('Распределение средней выручки: разовые vs вернувшиеся пользователи', 
          fontsize=16, fontweight='bold', color=font_color, pad=25)
plt.xlabel('Средняя выручка за заказ (руб.)', fontsize=12, color=font_color)
plt.ylabel('Плотность распределения', fontsize=12, color=font_color)

# Настройка осей
plt.xlim(-50, 2600)
plt.legend(frameon=False, fontsize=11)

# Управление сеткой
plt.grid(axis='y', alpha=0.15)
plt.grid(axis='x', visible=False)

sns.despine(left=True)
plt.show()

Анализ графика:
- Новые пользователи:
    - Наблюдается экстремально высокая концентрация заказов в диапазоне 0–200 рублей. Этот пик формируют разовые микропокупки (кино, бюджетные выставки) или заказы с применением промокодов.
    - Распределение имеет резкую правостороннюю асимметрию — плотность транзакций стремительно падает после отметки в 400 рублей. 
    - Наличие редких, очень дорогих разовых заказов в «хвосте» распределения (справа) вытягивает среднее значение до 546 рублей
- Вернувшиеся пользователи:
    -  Распределение имеет более куполообразный и пологий вид с высокой плотностью в диапазоне 400–1000 рублей. Это ядро монетизации сервиса.
    - У вернувшейся группы практически отсутствует аномальный пик в зоне сверхнизких чеков. Это говорит о том, что вернувшиеся клиенты ориентированы на стандартную стоимость услуг, либо в их профиле отсутствуют повторные покупки «за 1 рубль».

<b>Выводы:</b>
- Хотя средний чек обеих групп совпадает на уровне 545–546 руб., это сходство обманчиво и вызвано аномальными выбросами в группе новичков. 
- Типичный вернувшийся пользователь (медиана 496 руб.) на 31% «дороже» типичного новичка (медиана 378 руб.).
- У лояльных пользователей более устойчивый профиль трат (ниже std), а порог входа (25-й перцентиль) в два раза выше, чем у разовых клиентов.
-  Высокий чек первого заказа — это статистически подтвержденный индикатор того, что пользователь станет лояльным.



---

**Задача 4.2.2.** Сравните распределение по средней выручке с заказа в двух группах пользователей:

- совершившие 2–4 заказа;
- совершившие 5 и более заказов.

Ответьте на вопрос: есть ли различия по значению средней выручки с заказа между пользователями этих двух групп?

---


Сопоставим поведение двух групп вернувшихся пользователей: «активных» (2–4 заказа) и «суперактивных» (5+ заказов).

In [ ]:
# Разделение лояльных пользователей на две группы
few_orders = user_profiles_final[user_profiles_final['is_two'] == 1 & (user_profiles_final['is_five'] == 0)]['avg_revenue']
many_orders = user_profiles_final[user_profiles_final['is_five'] == 1]['avg_revenue']

In [ ]:
few_orders.describe()

In [ ]:
many_orders.describe()

In [ ]:
plt.figure(figsize=(15, 8))

# Построение гистограмм плотности
sns.histplot(few_orders, bins=50, stat="density", color='#aa7498', alpha=0.4, 
             label='Активные (2–4 заказа)', kde=True, edgecolor='none')
sns.histplot(many_orders, bins=50, stat="density", color='#621237', alpha=0.5, 
             label='Суперактивные (5+ заказов)', kde=True, edgecolor='none')

# Расчет средних арифметических
mean_few = few_orders.mean()
mean_many = many_orders.mean()

# Визуализация линий средних
plt.axvline(mean_few, color='#aa7498', linestyle='--', linewidth=2)
plt.axvline(mean_many, color='#621237', linestyle='--', linewidth=2)

# Аннотации для средних (координата y=0.0014 для стабильной видимости)
plt.text(mean_few - 30, 0.0014, f'Среднее (2-4): {mean_few:.0f} р.', 
         color='#aa7498', fontweight='bold', ha='right')
plt.text(mean_many + 30, 0.0014, f'Среднее (5+): {mean_many:.0f} р.', 
         color='#621237', fontweight='bold', ha='left')

# Оформление
plt.title('Распределение средней выручки: активные vs суперактивные пользователи', 
          fontsize=16, fontweight='bold', color=font_color, pad=25)
plt.xlabel('Средняя выручка за заказ (руб.)', fontsize=12, color=font_color)
plt.ylabel('Плотность распределения', fontsize=12, color=font_color)

plt.xlim(-50, 2600)
#plt.ylim(0, 0.0016) 
plt.legend(frameon=False, fontsize=11)
plt.grid(axis='y', alpha=0.15)
plt.grid(axis='x', visible=False)
sns.despine(left=True)

plt.show()

Анализ графика:
- Пользователи с 2–4 заказами (активные):
    - На графике сохраняется выраженный пик в диапазоне 0–200 рублей. Это указывает на то, что активные пользователи на начальном этапе лояльности продолжают совершать разовые бюджетные покупки (кино, выставки по акциям).
    - Средний чек 552 р. - значение выше, чем у суперактивных, из-за более высокого «хвоста» в правой части графика. Одиночные дорогие покупки в этой группе сильнее влияют на среднее арифметическое.
- Пользователи с 5+ заказами (Суперактивные):
    - Пик в зоне микрочеков практически исчез. Основная плотность распределения сосредоточена в «куполе» вокруг 500–600 рублей.
    - Средний чек 536 р. Несмотря на то, что среднее чуть ниже, распределение суперактивных пользователей гораздо более симметричное и предсказуемое.
    
<b>Выводы:</b>
- Основное различие между группами заключается в постепенном исчезновении сверхдешевых покупок (промокоды, билеты за 1 рубль) из профиля пользователя при переходе в суперактивную группу.
-  Обе кривые KDE практически идентичны в диапазоне 600–1200 рублей. Это подтверждает наличие устойчивого сегмента потребления (театры, средний ценовой сегмент концертов), который является целевым для всех лояльных категорий.
- Суперактивность выражается не в росте суммы среднего заказа, а в высокой частоте визитов при сохранении стабильного чека. Для бизнеса суперактивный клиент выгоднее за счет своей предсказуемости и отсутствия зависимости от промо-акций.

---

**Задача 4.2.3.** Проанализируйте влияние среднего количества билетов в заказе на вероятность повторной покупки.

- Изучите распределение пользователей по среднему количеству билетов в заказе (`avg_tickets_count`) и опишите основные наблюдения.
- Разделите пользователей на несколько сегментов по среднему количеству билетов в заказе:
    - от 1 до 2 билетов;
    - от 2 до 3 билетов;
    - от 3 до 5 билетов;
    - от 5 и более билетов.
- Для каждого сегмента подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы.
- Ответьте на вопросы:
    - Как распределены пользователи по сегментам — равномерно или сконцентрировано?
    - Есть ли сегменты с аномально высокой или низкой долей повторных покупок?

---

In [ ]:
# Применяем функцию сегментацию к профилям
user_profiles_final['ticket_segment'] = user_profiles_final['avg_tickets'].apply(get_ticket_segment)

In [ ]:
# Расчет статистики по сегментам
ticket_stats = user_profiles_final.groupby('ticket_segment').agg(
    users_count=('user_id', 'count'),
    retention_rate=('is_two', 'mean')
).reset_index()

In [ ]:
ticket_stats['share'] = (ticket_stats['users_count'] / len(user_profiles_final) * 100).round(1)

In [ ]:
ticket_stats

In [ ]:
# Визуализация влияния состава заказа на возврат

fig = plt.figure(figsize=(16, 8))

# Параметры сетки: 1 строка, 2 основных столбца и узкий разделитель между ними
gs = fig.add_gridspec(1, 3, width_ratios=[1, 0.05, 1])

# График 1. Распределение пользователей
ax1 = fig.add_subplot(gs[0, 0])
total_users = len(user_profiles_final)
ticket_counts = user_profiles_final['ticket_segment'].value_counts().sort_index()

sns.barplot(x=ticket_counts.index, y=ticket_counts.values, palette=colors, ax=ax1)

for p in ax1.patches:
    height = p.get_height()
    percentage = (height / total_users) * 100
    ax1.text(p.get_x() + p.get_width()/2., height + 100,
            f'{int(height):,}\n({percentage:.1f}%)', 
            ha="center", va="bottom", fontsize=11, fontweight='bold', color=font_color)

ax1.set_title('Распределение пользователей по сегментам', fontsize=15, fontweight='bold', pad=30, color=font_color)
ax1.set_xlabel('')
ax1.set_ylabel('')
ax1.set_yticks([])
ax1.set_ylim(0, ticket_counts.max() * 1.2)
sns.despine(ax=ax1, left=True)

# Разделительная линия 
ax_line = fig.add_subplot(gs[0, 1])
ax_line.axvline(x=0.5, color='#e0e0e0', linestyle='-', linewidth=1) # Светло-серая линия
ax_line.axis('off') # Скрываем оси разделителя

# График 2. Retention 
ax2 = fig.add_subplot(gs[0, 2])
avg_retention = user_profiles_final['is_two'].mean()
retention_data = user_profiles_final.groupby('ticket_segment')['is_two'].mean().sort_index()

sns.barplot(x=retention_data.index, y=retention_data.values, palette=colors, ax=ax2)

# Пунктирная линия среднего и подпись над ней
ax2.axhline(avg_retention, color=accent_color, linestyle='--', alpha=0.6)
ax2.text(3.4, avg_retention + 0.01, f'Среднее: {avg_retention:.1%}', 
         color=accent_color, fontweight='bold', fontsize=11, ha='right')

for p in ax2.patches:
    height = p.get_height()
    ax2.text(p.get_x() + p.get_width()/2., height + 0.01,
            f'{height:.1%}', 
            ha="center", va="bottom", fontsize=11, fontweight='bold', color=font_color)

ax2.set_title('Доля повторных покупок', fontsize=15, fontweight='bold', pad=30, color=font_color)
ax2.set_xlabel('')
ax2.set_ylabel('')
ax2.set_yticks([])
ax2.set_ylim(0, 0.85)
sns.despine(ax=ax2, left=True)

plt.tight_layout()
plt.show()

Анализ графика:
- Распределение пользователей по сегментам
    - Основная масса пользователей (85.8%) сосредоточена в сегментах «2-3 билета» (43.9%) и «3-5 билетов» (41.9%). Это подтверждает социальный характер сервиса: абсолютное большинство людей планирует досуг минимум для двоих, а почти половина — для небольших компаний или семей.
    -  Одиночные пользователи (1-2 билета) составляют лишь 11.2% аудитории. «Организаторы» крупных групп (от 5 билетов) — самый малочисленный сегмент (3%).
- Доля повторных покупок
    - Пользователи, покупающие 2-3 билета, демонстрируют феноменальную лояльность — 73.6% возврата. Это на 12.3 п.п. выше среднего уровня (61.3%). Данный сегмент является фундаментом лояльной базы Яндекс Афиши.
    -  Сегмент «от 5 билетов» показывает критически низкое удержание — всего 17.7%. Это подтверждает гипотезу о «разовом событии»: такие заказы чаще всего приурочены к празднику или корпоративному выходу, после чего организатор не видит смысла возвращаться к сервису.
    - Пользователи, покупающие 1-2 билета, возвращаются реже среднего (51.3%). Отсутствие социальной составляющей (компании) снижает мотивацию к повторному использованию платформы.
    

<b>Выводы:</b> лояльность на Яндекс Афише напрямую зависит от количества билетов в первом заказе, но эта зависимость нелинейная. Максимальную отдачу дают «парные» посещения (2-3 билета). Как только компания становится больше 5 человек, вероятность удержания пользователя резко падает.


---

#### 4.3. Исследование временных характеристик первого заказа и их влияния на повторные покупки

Изучите временные параметры, связанные с первым заказом пользователей:

- день недели первой покупки;
- время с момента первой покупки — лайфтайм;
- средний интервал между покупками пользователей с повторными заказами.

---

**Задача 4.3.1.** Проанализируйте, как день недели, в которой была совершена первая покупка, влияет на поведение пользователей.

- По данным даты первого заказа выделите день недели.
- Для каждого дня недели подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы. Результаты визуализируйте.
- Ответьте на вопрос: влияет ли день недели, в которую совершена первая покупка, на вероятность возврата клиента?

---


In [ ]:
# Извлекаем день недели из даты первого заказа
user_profiles_final['first_order_weekday'] = user_profiles_final['first_order_date'].dt.day_name()

In [ ]:
user_profiles_final.head()

In [ ]:
# Определяем порядок дней для корректного отображения на графиках
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

In [ ]:
# Группировка данных
weekday_stats = user_profiles_final.groupby('first_order_weekday').agg(
    users_count=('user_id', 'count'),
    retention_rate=('is_two', 'mean')
).reindex(weekday_order).reset_index()

In [ ]:
fig = plt.figure(figsize=(16, 6))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 0.05, 1])
avg_retention = user_profiles_final['is_two'].mean()

# График 1. Распределение по дням недели
ax1 = fig.add_subplot(gs[0])
sns.barplot(data=weekday_stats, x='first_order_weekday', y='users_count', palette=colors, ax=ax1)

for p in ax1.patches:
    height = p.get_height()
    percentage = (height / len(user_profiles_final)) * 100
    ax1.text(p.get_x() + p.get_width()/2., height + 50,
            f'{int(height):,}\n({percentage:.1f}%)', 
            ha="center", va="bottom", fontsize=10, fontweight='bold', color=font_color)

ax1.set_title('Распределение первых заказов по дням недели', fontsize=14, fontweight='bold', pad=30, color=font_color)
ax1.set_xticklabels(['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс'])
ax1.set_xlabel('')
ax1.set_ylabel('')
ax1.set_yticks([])
sns.despine(ax=ax1, left=True)

# Разделительная линия 
ax_line = fig.add_subplot(gs[1])
ax_line.axvline(x=0.5, color='#e0e0e0', linestyle='-', linewidth=1)
ax_line.axis('off')

# График 2. Retention  по дням недели 
ax2 = fig.add_subplot(gs[2])
sns.barplot(data=weekday_stats, x='first_order_weekday', y='retention_rate', palette=colors, ax=ax2)

# Линия среднего и подпись
ax2.axhline(avg_retention, color=accent_color, linestyle='--', alpha=0.6)
ax2.text(6.4, avg_retention + 0.08, f'Среднее: {avg_retention:.1%}', 
         color=accent_color, fontweight='bold', fontsize=11, ha='right')

for p in ax2.patches:
    height = p.get_height()
    ax2.text(p.get_x() + p.get_width()/2., height + 0.005,
            f'{height:.1%}', 
            ha="center", va="bottom", fontsize=10, fontweight='bold', color=font_color)

ax2.set_title('Возвращаемость в зависимости от дня входа', fontsize=14, fontweight='bold', pad=30, color=font_color)
ax2.set_xticklabels(['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс'])
ax2.set_xlabel('')
ax2.set_ylabel('')
ax2.set_yticks([])
ax2.set_ylim(0, 0.75)
sns.despine(ax=ax2, left=True)

plt.tight_layout()
plt.show()

Анализ графика:
- Распределение первых заказов по дням недели
    - Привлечение новых пользователей распределено по неделе достаточно стабильно, без резких провалов. Разница между самым активным и самым пассивным днем составляет всего 2.5% п.п.
    -  Наибольшее количество первых покупок совершается в субботу (15.3%) и пятницу (15.1%). Это ожидаемо для сервиса развлечений, так как пользователи активнее всего ищут досуг непосредственно перед выходными или во время них.
    - Самый низкий показатель привлечения зафиксирован в воскресенье (12.8%). Вероятно, к этому моменту планы на отдых уже реализованы, и пользователи меньше склонны к спонтанным новым покупкам.
- Возвращаемость в зависимости от дня входа
    - Доля повторных покупок практически не зависит от дня недели, в который был совершен первый заказ. Все значения колеблются в узком диапазоне вокруг среднего уровня 61.3%
    - Максимальный разброс между днями (от 59.5% в четверг до 62.8% в субботу) составляет всего 3.3% п.п.:  день недели не является фактором, определяющим будущую лояльность клиента.
    - Незначительное преимущество по возвращаемости имеют пользователи, пришедшие в субботу (62.8%), понедельник (63%) и среду (62.2%).
    
    
<b>Вывод:</b> день недели, в который пользователь впервые воспользовался Яндекс Афишей, является нейтральным фактором для анализа лояльности.

---

**Задача 4.3.2.** Изучите, как средний интервал между заказами влияет на удержание клиентов.

- Рассчитайте среднее время между заказами для двух групп пользователей:
    - совершившие 2–4 заказа;
    - совершившие 5 и более заказов.
- Исследуйте, как средний интервал между заказами влияет на вероятность повторного заказа, и сделайте выводы.

---


Проанализируем временные интервалы между покупками

In [ ]:
# Рассчитаем средний интервал для двух целевых групп
few_orders_interval = user_profiles_final[(user_profiles_final['is_two'] == 1) \
                                          & (user_profiles_final['is_five'] == 0)]['avg_time_diff']
many_orders_interval = user_profiles_final[user_profiles_final['is_five'] == 1]['avg_time_diff']

In [ ]:
# описательная статистика для группы 2-4 заказа
few_orders_interval.describe()

In [ ]:
# описательная статистика для группы 5+ заказов
many_orders_interval.describe()

In [ ]:
print(f"Средний интервал (2-4 заказа): {few_orders_interval.mean():.1f} дн.")
print(f"Средний интервал (5+ заказов): {many_orders_interval.mean():.1f} дн.")

In [ ]:
plt.figure(figsize=(15, 8))

# Расчет средних
mean_few = few_orders_interval.mean()
mean_many = many_orders_interval.mean()

# Построение гистограмм плотности
sns.histplot(few_orders_interval, bins=100, stat="density", color='#aa7498', alpha=0.4, 
             label='Активные (2–4 заказа)', kde=True, edgecolor='none')
sns.histplot(many_orders_interval, bins=100, stat="density", color='#621237', alpha=0.5, 
             label='Суперактивные (5+ заказов)', kde=True, edgecolor='none')

# Линии средних
plt.axvline(mean_few, color='#aa7498', linestyle='--', linewidth=2)
plt.axvline(mean_many, color='#621237', linestyle='--', linewidth=2)

# Подписи для средних
plt.text(mean_few + 1, 0.17, f'Среднее (2-4): {mean_few:.1f} дн.', 
         color='#aa7498', fontweight='bold', ha='left')
plt.text(mean_many + 0.8, 0.17, f'Среднее (5+): {mean_many:.1f} дн.', 
         color='#621237', fontweight='bold', ha='right')

# Оформление
plt.title('Распределение интервалов между покупками', fontsize=16, fontweight='bold', color=font_color, pad=25)
plt.xlabel('Среднее количество дней между заказами', fontsize=12, color=font_color)
plt.ylabel('Плотность распределения', fontsize=12, color=font_color)

plt.xlim(-1, 61) 
plt.legend(frameon=False)
plt.grid(axis='y', alpha=0.15)
plt.grid(axis='x', visible=False)
sns.despine(left=True)

plt.show()

Анализ графика:
- Активные пользователи (2–4 заказа)
    - Средний интервал составляет 21.3 дня, в то время как медиана — всего 9 дней. Огромное стандартное отклонение (28.3) и максимальное значение в 148 дней указывают на отсутствие регулярной привычки. В этой группе много «стихийных» возвратов с огромными паузами.
    - Значение 25-го перцентиля равно 0. Это значит, что как минимум четверть этой группы — люди, которые докупают билеты в тот же день, но не обязательно возвращаются за новым мероприятием позже.
- Суперактивные пользователи (5+ заказов)
    -  Среднее (9.9) и медиана (8.1) находятся очень близко друг к другу. Низкое стандартное отклонение (7.8) и максимальный интервал всего в 37.5 дней говорят о дисциплинированном и предсказуемом поведении.
    - Эти пользователи возвращаются системно. 75% их заказов совершаются с интервалом до 14 дней. Это «живое» ядро сервиса, которое встроено в их еженедельный цикл досуга.
    
<b>Вывод: </b>если средний интервал пользователя начинает превышать 14 дней (75-й перцентиль суперактивных), он с высокой долей вероятности никогда не станет частью высокодоходного ядра и останется «эпизодическим» клиентом.

---

#### 4.4. Корреляционный анализ количества покупок и признаков пользователя

Изучите, какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок. Для этого используйте универсальный коэффициент корреляции `phi_k`, который позволяет анализировать как числовые, так и категориальные признаки.

---

**Задача 4.4.1:** Проведите корреляционный анализ:
- Рассчитайте коэффициент корреляции `phi_k` между признаками профиля пользователя и числом заказов (`total_orders`). При необходимости используйте параметр `interval_cols` для определения интервальных данных.
- Проанализируйте полученные результаты. Если полученные значения будут близки к нулю, проверьте разброс данных в `total_orders`. Такое возможно, когда в данных преобладает одно значение: в таком случае корреляционный анализ может показать отсутствие связей. Чтобы этого избежать, выделите сегменты пользователей по полю `total_orders`, а затем повторите корреляционный анализ. Выделите такие сегменты:
    - 1 заказ;
    - от 2 до 4 заказов;
    - от 5 и выше.
- Визуализируйте результат корреляции с помощью тепловой карты.
- Ответьте на вопрос: какие признаки наиболее связаны с количеством заказов?

---

In [ ]:
user_profiles_final.head()

In [ ]:
# Создаем сегменты по количеству заказов
def get_order_segment(orders):
    if orders == 1:
        return '1 заказ'
    elif 2 <= orders <= 4:
        return '2-4 заказа'
    else:
        return '5+ заказов'

In [ ]:
user_profiles_final['order_segment'] = user_profiles_final['total_orders'].apply(get_order_segment)

In [ ]:
# Выбираем признаки для анализа 
corr_features = [
    'first_device',        # Устройство
    'first_region',        # регион
    'first_service',       # сервис
    'order_segment',       # Целевой признак
    'first_genre',         # Тип мероприятия
    'ticket_segment',      # Состав заказа
    'is_promo_first',      # Платный/Промо
    'first_order_weekday', # День недели 
    'first_order_revenue'  # Чек первой покупки
]

In [ ]:
# Расчет матрицы корреляции phik
# Указываем числовые колонки в interval_cols
#interval_cols = ['avg_revenue']
#phik_matrix = user_profiles_final[corr_features].phik_matrix(interval_cols=interval_cols)
phik_matrix = user_profiles_final[corr_features].phik_matrix(interval_cols=['first_order_revenue'])

In [ ]:
plt.figure(figsize=(12, 10))

sns.heatmap(
    phik_matrix, 
    annot=True, 
    fmt=".2f", 
    cmap=my_cmap, 
    linewidths=0.5,
    cbar_kws={"shrink": .8}
)

plt.title('Корреляция признаков с сегментами заказов ($\phi_k$)', 
          fontsize=16, fontweight='bold', color=font_color, pad=20)
plt.show()

In [ ]:
order_corr = phik_matrix['order_segment'].sort_values(ascending=False)

print("Корреляция признаков с количеством заказов (order_segment) ---")
print(order_corr.drop('order_segment')) # удаляем самокорреляцию 1.0

Анализ связей с целевым признаком (order_segment)
- Самая сильная связь — ticket_segment (0.23). Состав первого заказа (количество билетов) является ключевым предиктором лояльности. Поведенческий паттерн («иду один», «иду парой» или «веду компанию») сильнее всего коррелирует с тем, вернется ли пользователь в сервис.
- Умеренная связь — first_region (0.12). География входа имеет значение. Это подтверждает выводы о «сильных» регионах (Шанырский) и «зонах риска», где специфика локального рынка влияет на частоту покупок.
- Слабая связь:
    - first_service (0.08). Билетный оператор влияет на пользовательский опыт, но не является решающим фактором для формирования долгосрочной привычки.
    - first_genre (0.04). Тип первого мероприятия важен для входа, но само количество последующих заказов от него зависит слабо.
    - first_order_weekday (0.03). День недели не определяет лояльность.
    - first_device (0.02).Тип устройства не определяет количество покупок в будущем.
    - is_promo_first (0.01). Промокоды на первом заказе никак не коррелируют с итоговым количеством заказов. Они одинаково присутствуют и у «разовых» пользователей, и у суперактивных.
    - first_order_revenue (0.00). Сумма первого чека абсолютно не влияет на то, станет ли пользователь лояльным.
    
Внутренние зависимости (мультиколлинеарность)
- first_region и first_service (0.70). Почти линейная связь. Это говорит о жесткой привязке определенных билетных партнеров к конкретным регионам (региональная монополия).
- first_genre со service (0.59) и region (0.51): Типы мероприятий сильно зависят от того, в каком городе живет пользователь и какие партнеры там работают.
- first_order_revenue и service/region (0.43–0.46): Средний чек первого заказа диктуется скорее регионом проживания и ценовой политикой местного партнера, чем желанием пользователя потратить больше.

### 5. Общий вывод и рекомендации

В конце проекта напишите общий вывод и рекомендации: расскажите заказчику, на что нужно обратить внимание. В выводах кратко укажите:

- **Информацию о данных**, с которыми вы работали, и то, как они были подготовлены: например, расскажите о фильтрации данных, переводе тенге в рубли, фильтрации выбросов.
- **Основные результаты анализа.** Например, укажите:
    - Сколько пользователей в выборке? Как распределены пользователи по числу заказов? Какие ещё статистические показатели вы подсчитали важным во время изучения данных?
    - Какие признаки первого заказа связаны с возвратом пользователей?
    - Как связаны средняя выручка и количество билетов в заказе с вероятностью повторных покупок?
    - Какие временные характеристики влияют на удержание (день недели, интервалы между покупками)?
    - Какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок согласно результатам корреляционного анализа?
- Дополните выводы информацией, которая покажется вам важной и интересной. Следите за общим объёмом выводов — они должны быть компактными и ёмкими.

В конце предложите заказчику рекомендации о том, как именно действовать в его ситуации. Например, укажите, на какие сегменты пользователей стоит обратить внимание в первую очередь, а какие нуждаются в дополнительных маркетинговых усилиях.

<b>ОБЩИЙ ВЫВОД ПО ПРОЕКТУ</b>

1. Подготовка данных и фильтрация. 

- Работа проведена с датасетом из 290 611 строк. В ходе предобработки выполнены следующие действия:
    - Валютная конвертация. Выручка приведена к единому стандарту в рублях.
    - Очистка от аномалий. Удалены технические ошибки (отрицательная выручка) и экстремальные выбросы по 99-му перцентилю (заказы более 2800 руб. и более 12 билетов). Это позволило исключить перекупщиков и корпоративные заказы, искажающие поведение обычных людей.
    - Сохранение промо-заказов. Покупки менее 10 руб. намеренно оставлены для анализа влияния скидок на удержание.
    - Итоговая выборка: Составила 21 612 уникальных профилей пользователей (потеря данных после всех фильтраций — менее 4%, что гарантирует репрезентативность).
    
2. Основные результаты анализа
- Лояльность. Доля пользователей, совершивших 2 и более заказа (Retention Rate), составляет 61.3%. Группа «суперлояльных» (5+ заказов) занимает 28.9% базы.
- Портрет пользователя. Средний клиент покупает 2.7 билета с чеком около 545 руб.
- Финансовое поведение. Выявлен качественный разрыв. Типичный лояльный пользователь (медиана 496 руб.) на 31% «дороже» типичного новичка (медиана 378 руб.). Рост лояльности сопровождается отказом от микрочеков и переходом к осознанным покупкам.

3. Факторы возврата и удержания
    - Тип мероприятия. «Интеллектуальные» точки входа — Выставки (64.4%) и Театр (63.4%) — удерживают лучше всего. Спорт (55.8%) — самый «разовый» жанр.
    - Состав заказа. Выявлена нелинейная зависимость. «Пары» (2-3 билета) — самый лояльный сегмент (73.6%). Одиночки и большие группы (5+) возвращаются значительно реже.
    - Временные параметры. День недели первого заказа не влияет на лояльность. Однако критически важен интервал между покупками: у суперактивных пользователей он стабилен и составляет 10 дней. Рост интервала свыше 14 дней — сигнал к скорому оттоку.
    - Тип входа. Вопреки ожиданиям, Промо-вход (<10р) показал аномально высокую лояльность (64.9%). Скидка на первый заказ является эффективным инструментом «привязки» качественной аудитории.

4. Корреляционный анализ  ($\phi_k$)
- Математически доказано, что количество заказов сильнее всего связано с составом первого заказа (ticket_segment, коэфф. 0.23). Региональная специфика также имеет значение (0.12). При этом сумма первого чека и устройство практически не коррелируют с итоговой лояльностью.
    

<b>РЕКОМЕНДАЦИИ ДЛЯ ЗАКАЗЧИКА</b>
- Пользователи, покупающие 2-3 билета, — фундамент бизнеса. Рекомендуется внедрить механику «Второй билет в подарок» или скидки на парные места для других категорий, чтобы закрепить этот паттерн.
- Выставки и театры приводят самых лояльных клиентов. Стоит увеличить маркетинговый бюджет на продвижение этих категорий как «первого шага» в сервисе.
- Сегмент "5+ билетов" сейчас разовый (Retention 17.7%). Необходимо разработать программу лояльности для «лидеров компаний» (например, кешбэк баллами Плюса за большой объем билетов), чтобы стимулировать их возвращаться именно в Яндекс Афишу при следующем сборе друзей.
- Mobile дает 83% трафика, но удерживает хуже, чем Desktop. Требуется аудит UX мобильной версии/приложения с целью упростить повторную покупку (сохранение предпочтений, быстрые рекомендации).
- Использовать опыт Шанырского региона (Retention 67%) как эталон для развития других областей. Проанализировать, какие партнеры или типы событий обеспечивают там такой аномальный возврат.
- Чтобы вырастить лояльную базу, Яндекс Афише нужно фокусироваться на социальном контексте покупки (сколько билетов берет человек) и региональной специфике, а не на сумме первого чека или раздаче промокодов. Промокоды приводят людей, но их превращение в лояльных клиентов зависит уже от других факторов.

### 6. Финализация проекта и публикация в Git

Когда вы закончите анализировать данные, оформите проект, а затем опубликуйте его.

Выполните следующие действия:

1. Создайте файл `.gitignore`. Добавьте в него все временные и чувствительные файлы, которые не должны попасть в репозиторий.
2. Сформируйте файл `requirements.txt`. Зафиксируйте все библиотеки, которые вы использовали в проекте.
3. Вынести все чувствительные данные (параметры подключения к базе) в `.env`файл.
4. Проверьте, что проект запускается и воспроизводим.
5. Загрузите проект в публичный репозиторий — например, на GitHub. Убедитесь, что все нужные файлы находятся в репозитории, исключая те, что в `.gitignore`. Ссылка на репозиторий понадобится для отправки проекта на проверку. Вставьте её в шаблон проекта в тетрадке Jupyter Notebook перед отправкой проекта на ревью.

**Вставьте ссылку на проект в этой ячейке тетрадки перед отправкой проекта на ревью.**